# CATI — Singapore Smart City Traffic Analytics
## Context-Aware Traffic Intelligence Demo

End-to-end pipeline on live Singapore LTA camera feeds:

| Stage | Module | What it does |
|---|---|---|
| Detection | `CATIPipeline` | FiLM-conditioned YOLOv11 — weather/time/location aware |
| Tracking | `SingaporeTracker` | Direction-constrained ByteTrack per expressway |
| Re-ID | `VehicleReID` | OSNet-x0.25 cross-camera appearance matching |
| Analytics | `TrafficAnalytics` | Occupancy, LOS A–F, congestion score |
| Speed | `SpeedEstimator` | Inter-camera speed via GPS edge distances |
| Network | `CameraNetwork` | 90-camera expressway graph (75 edges) |

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_DIR = '/content/sg-smart-city-analytics'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Suhxs-Reddy/sg-smart-city-analytics.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
    for k in list(sys.modules.keys()):
        if k.startswith('src.'): del sys.modules[k]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

!pip install -q ultralytics torch torchvision scipy pillow opencv-python-headless torchreid

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}  ({gpu.total_memory/2**30:.1f} GB)')

MODEL_DIR   = '/content/drive/MyDrive/sg_smart_city/models'
FEATURE_DIR = '/content/drive/MyDrive/sg_smart_city/data/features'
YOLO_DIR    = '/content/drive/MyDrive/sg_smart_city/data/yolo_dataset'
OUTPUT_DIR  = '/content/drive/MyDrive/sg_smart_city/demo_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

from pathlib import Path
PHASE2_DIR  = Path(MODEL_DIR) / 'phase2'
YOLO_WEIGHTS = str(max(PHASE2_DIR.glob('**/best.pt'), key=lambda p: p.stat().st_mtime))
CATI_WEIGHTS = str(max(
    list(PHASE2_DIR.glob('**/cati_phase2_final.pt')) +
    list(PHASE2_DIR.glob('**/cati_phase2_epoch*.pt')),
    key=lambda p: p.stat().st_mtime
))
print(f'YOLO: {YOLO_WEIGHTS}')
print(f'CATI: {CATI_WEIGHTS}')

In [ ]:
# ── Cell 2: Camera Network Overview ───────────────────────────────────────
from src.analytics.camera_network import CameraNetwork
from src.analytics.camera_map import CameraMap

net = CameraNetwork()
print(net.summary())
print()
print('CTE corridor (S→N):')
for e in net.adjacent_pairs('CTE')[:6]:
    print(f'  {e.cam_a.camera_id} ({e.cam_a.area}) → {e.cam_b.camera_id} ({e.cam_b.area})  {e.distance_km:.2f} km')
print('  ...')

In [ ]:
# ── Cell 3: Load Pipeline ──────────────────────────────────────────────────
from src.inference import CATIPipeline

pipeline = CATIPipeline(
    yolo_weights=YOLO_WEIGHTS,
    cati_weights=CATI_WEIGHTS,
    feature_dir=FEATURE_DIR,
    device='cuda',
    conf=0.25,
    use_neck_film=True,
)
print('Pipeline loaded.')

In [ ]:
# ── Cell 4: Fetch Live LTA Frames ─────────────────────────────────────────
# Pulls the current frame from every camera in real-time.
import urllib.request, json, cv2, numpy as np
from datetime import datetime, timezone, timedelta

SGT = timezone(timedelta(hours=8))

data = json.loads(urllib.request.urlopen(
    'https://api.data.gov.sg/v1/transport/traffic-images'
).read())
cameras = data['items'][0]['cameras']
timestamp = datetime.now(SGT).isoformat()

print(f'Fetched {len(cameras)} cameras at {timestamp}')

# Download frames for a representative set of cameras across expressways
SELECTED = ['1001','1701','4701','4710','5794','6701','7791','8701','9701']
frames = {}
for cam in cameras:
    cid = cam['camera_id']
    if cid not in SELECTED:
        continue
    try:
        resp = urllib.request.urlopen(cam['image'], timeout=5)
        img_bytes = np.frombuffer(resp.read(), np.uint8)
        img = cv2.imdecode(img_bytes, cv2.IMREAD_COLOR)
        if img is not None:
            frames[cid] = {'image': img, 'lat': cam['location']['latitude'],
                           'lon': cam['location']['longitude']}
            print(f'  {cid}: {img.shape[1]}x{img.shape[0]}')
    except Exception as e:
        print(f'  {cid}: failed ({e})')

print(f'\n{len(frames)} frames ready for inference')

In [ ]:
# ── Cell 5: Fetch Live Weather ────────────────────────────────────────────
import urllib.request, json

def get_live_weather():
    try:
        data = json.loads(urllib.request.urlopen(
            'https://api.data.gov.sg/v1/environment/24-hour-weather-forecast', timeout=5
        ).read())
        return data['items'][0]['general']['forecast']
    except Exception:
        return 'unknown'

def get_live_temperature():
    try:
        data = json.loads(urllib.request.urlopen(
            'https://api.data.gov.sg/v1/environment/air-temperature', timeout=5
        ).read())
        readings = data['items'][0]['readings']
        return round(sum(r['value'] for r in readings) / len(readings), 1)
    except Exception:
        return 28.0

weather = get_live_weather()
temperature = get_live_temperature()
print(f'Weather: {weather}')
print(f'Temperature: {temperature}°C')

In [ ]:
# ── Cell 6: Run Inference on All Selected Cameras ─────────────────────────
import json
from datetime import datetime, timezone, timedelta

SGT = timezone(timedelta(hours=8))
timestamp = datetime.now(SGT).isoformat()

results = {}
for camera_id, frame_data in frames.items():
    result = pipeline.process_frame(
        image_bgr=frame_data['image'],
        camera_id=camera_id,
        timestamp=timestamp,
        weather=weather,
        temperature=temperature,
        frame_id=0,
    )
    results[camera_id] = result
    ts = result.traffic_state
    print(
        f'{camera_id:<6} {result.road:<5} {result.area:<20} '
        f'vehicles={ts.total_vehicles:<4} '
        f'occ={ts.occupancy*100:4.1f}% '
        f'LOS={ts.los.value} '
        f'congestion={ts.congestion_score:.2f} '
        f'[{ts.weather[:18]}]'
    )

print(f'\nInference complete. Avg pipeline: '
      f'{sum(r.pipeline_ms for r in results.values())/len(results):.0f}ms/frame')

In [ ]:
# ── Cell 7: Multi-Camera Speed Estimation (2 frames per camera) ───────────
# Run a second pass so the tracker accumulates tracks and re-ID can match
# vehicles across adjacent cameras on the same expressway.
print('Second pass (building re-ID gallery and cross-camera matches)...')

for camera_id, frame_data in frames.items():
    result = pipeline.process_frame(
        image_bgr=frame_data['image'],
        camera_id=camera_id,
        timestamp=timestamp,
        weather=weather,
        temperature=temperature,
        frame_id=1,
    )
    results[camera_id] = result  # update with frame-2 result
    if result.speed_readings:
        for sr in result.speed_readings:
            print(f'  SPEED: {sr.camera_from}→{sr.camera_to}  '
                  f'{sr.speed_kmh:.1f} km/h  [{sr.congestion_band}]  '
                  f'(limit {sr.speed_limit} km/h, sim={sr.similarity:.3f})')

print()
print('Road speed profile:')
for road, info in pipeline.speed_estimator.road_speed_profile().items():
    print(f'  {road:<6} {info["avg_speed_kmh"]:>6.1f} km/h  '
          f'[{info["congestion_band"]}]  '
          f'limit {info["speed_limit"]} km/h  '
          f'({info["num_readings"]} readings)')

In [ ]:
# ── Cell 8: Annotated Output Frames ───────────────────────────────────────
import cv2
from IPython.display import display, Image as IPImage
import os

for camera_id, result in results.items():
    frame = frames[camera_id]['image']
    annotated = pipeline.draw(frame, result)

    out_path = f'{OUTPUT_DIR}/{camera_id}_{result.road}_{result.traffic_state.los.value}.jpg'
    cv2.imwrite(out_path, annotated)

    # Display inline (resize for Colab)
    h, w = annotated.shape[:2]
    scale = min(1.0, 900 / w)
    disp = cv2.resize(annotated, (int(w*scale), int(h*scale)))
    _, buf = cv2.imencode('.jpg', disp, [cv2.IMWRITE_JPEG_QUALITY, 85])
    print(f'Camera {camera_id} | {result.road} | {result.area} | LOS {result.traffic_state.los.value}')
    display(IPImage(data=buf.tobytes()))
    print()

In [ ]:
# ── Cell 9: Network Summary Dashboard ─────────────────────────────────────
import json
from src.analytics.traffic_analytics import TrafficAnalytics

all_states = [r.traffic_state for r in results.values()]
summary = TrafficAnalytics().network_summary(all_states)

print('=' * 60)
print('  SINGAPORE EXPRESSWAY NETWORK SUMMARY')
print('=' * 60)
print(f'  Cameras active:  {summary["total_cameras"]}')
print(f'  Total vehicles:  {summary["total_vehicles"]}')
print(f'  Avg occupancy:   {summary["avg_occupancy"]*100:.1f}%')
print(f'  Avg congestion:  {summary["avg_congestion"]:.3f}')
print(f'  Worst camera:    {summary["worst_camera"]}')
print()
print('  By Road:')
for road, info in summary['by_road'].items():
    print(f'    {road:<8} {info["total_vehicles"]:>4} vehicles  '
          f'congestion={info["avg_congestion"]:.2f}  '
          f'LOS {info["avg_los"]}')
print()
print('  By Region:')
for region, info in summary['by_region'].items():
    print(f'    {region:<10} {info["cameras"]} cameras  '
          f'{info["total_vehicles"]:>4} vehicles  '
          f'congestion={info["avg_congestion"]:.2f}')

# Save full results to Drive
with open(f'{OUTPUT_DIR}/network_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'\nSaved to {OUTPUT_DIR}/network_summary.json')

In [ ]:
# ── Cell 10: Per-Camera Detail ────────────────────────────────────────────
print(f'{"Cam":<6} {"Road":<6} {"Region":<10} {"Area":<22} {"Veh":>4} {"Occ%":>6} {"LOS":<4} {"Cong":>6} {"Tracks":>7}')
print('-' * 80)
for cid, result in sorted(results.items()):
    ts = result.traffic_state
    print(
        f'{cid:<6} {result.road:<6} {result.region:<10} {result.area:<22} '
        f'{ts.total_vehicles:>4} {ts.occupancy*100:>5.1f}% '
        f'{ts.los.value:<4} {ts.congestion_score:>6.3f} '
        f'{result.num_active_tracks:>7}'
    )